In [ ]:
!pip install scikit-fuzzy

  Using cached scikit_fuzzy-0.5.0-py2.py3-none-any.whl.metadata (2.6 kB)
Using cached scikit_fuzzy-0.5.0-py2.py3-none-any.whl (920 kB)


In [ ]:
import numpy as np
import pandas as pd
import skfuzzy as fuzz
from skfuzzy import control as ctrl

In [ ]:
df = pd.read_csv("/content/Crop_recommendation.csv")

In [ ]:
# Input variables
temperature = ctrl.Antecedent(np.arange(0, 51, 1), 'temperature')
humidity = ctrl.Antecedent(np.arange(0, 101, 1), 'humidity')
rainfall = ctrl.Antecedent(np.arange(0, 301, 1), 'rainfall')

# Output variable
crop = ctrl.Consequent(np.arange(0, 3, 1), 'crop')
crop.defuzzify_method = 'centroid'

# Membership functions
temperature['low'] = fuzz.trimf(temperature.universe, [0, 0, 20])
temperature['medium'] = fuzz.trimf(temperature.universe, [15, 25, 35])
temperature['high'] = fuzz.trimf(temperature.universe, [30, 50, 50])

humidity['low'] = fuzz.trimf(humidity.universe, [0, 0, 40])
humidity['medium'] = fuzz.trimf(humidity.universe, [30, 50, 70])
humidity['high'] = fuzz.trimf(humidity.universe, [60, 100, 100])

rainfall['low'] = fuzz.trimf(rainfall.universe, [0, 0, 100])
rainfall['medium'] = fuzz.trimf(rainfall.universe, [80, 150, 220])
rainfall['high'] = fuzz.trimf(rainfall.universe, [200, 300, 300])

# Output membership
crop['wheat'] = fuzz.trimf(crop.universe, [0, 0, 1])
crop['maize'] = fuzz.trimf(crop.universe, [0, 1, 2])
crop['rice'] = fuzz.trimf(crop.universe, [1, 2, 2])

# Rules (improved coverage)
rule1 = ctrl.Rule(temperature['high'] & humidity['high'], crop['rice'])
rule2 = ctrl.Rule(rainfall['high'], crop['rice'])

rule3 = ctrl.Rule(temperature['medium'] & humidity['medium'], crop['maize'])
rule4 = ctrl.Rule(rainfall['medium'], crop['maize'])

rule5 = ctrl.Rule(temperature['low'], crop['wheat'])
rule6 = ctrl.Rule(rainfall['low'], crop['wheat'])

# Control system
crop_ctrl = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule5, rule6])
crop_simulation = ctrl.ControlSystemSimulation(crop_ctrl)

# ---- USER INPUT ----
temp = float(input("Enter temperature (0-50): "))
hum = float(input("Enter humidity (0-100): "))
rain = float(input("Enter rainfall (0-300): "))

crop_simulation.input['temperature'] = temp
crop_simulation.input['humidity'] = hum
crop_simulation.input['rainfall'] = rain

# Compute
crop_simulation.compute()

# Safe output handling
if 'crop' in crop_simulation.output:
    result = crop_simulation.output['crop']
else:
    print("No crop recommendation possible. Try different values.")
    exit()

# Result interpretation
if result < 0.5:
    print("🌱 Recommended Crop: Wheat")
elif result < 1.5:
    print("🌽 Recommended Crop: Maize")
else:
    print("🌾 Recommended Crop: Rice")

Enter temperature (0-50): 37
Enter humidity (0-100): 67
Enter rainfall (0-300): 200
🌽 Recommended Crop: Maize
